# 02. Feature Extraction

## Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import time
import json
from tqdm.auto import tqdm

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms

from sklearn.preprocessing import StandardScaler
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"[02-FEATURE-EXTRACTION] Libraries loaded")
print(f"[02-FEATURE-EXTRACTION] Device: {device}")
print(f"[02-FEATURE-EXTRACTION] PyTorch version: {torch.__version__}")

## Configuration

In [ ]:
# Input paths (from 01-Preprocessing-Step)
INPUT_DIR = '/kaggle/input/01-preprocessing-result'

# Fallback for local testing
if not os.path.exists(INPUT_DIR):
    INPUT_DIR = './01-preprocessing-output'

# Dataset base path
DATASET_BASE_PATH = '/kaggle/input/dataset-sampah'

if not os.path.exists(DATASET_BASE_PATH):
    DATASET_BASE_PATH = input("Enter dataset path: ").strip()

# Output directory
OUTPUT_DIR = './02-feature-extraction-output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Class configuration
CLASSES = ['Organik', 'Anorganik', 'Lainnya']

print(f"[02-FEATURE-EXTRACTION] Input directory: {INPUT_DIR}")
print(f"[02-FEATURE-EXTRACTION] Dataset path: {DATASET_BASE_PATH}")
print(f"[02-FEATURE-EXTRACTION] Output directory: {OUTPUT_DIR}")

## Load CSV Files from 01-Preprocessing

In [ ]:
print("[02-FEATURE-EXTRACTION] Loading CSV files from 01-Preprocessing...")

train_df = pd.read_csv(f'{INPUT_DIR}/train_dataset.csv')
val_df = pd.read_csv(f'{INPUT_DIR}/val_dataset.csv')
test_df = pd.read_csv(f'{INPUT_DIR}/test_dataset.csv')

print(f"[02-FEATURE-EXTRACTION] CSV files loaded:")
print(f"  Train: {len(train_df):,} samples")
print(f"  Val:   {len(val_df):,} samples")
print(f"  Test:  {len(test_df):,} samples")

print(f"\nDataFrame preview:")
print(train_df.head())

## Image Transforms (PyTorch Best Practice)

In [ ]:
# ImageNet normalization constants
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Training transform WITH augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),  # Auto-converts [0-255] to [0-1]
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Val/Test transform WITHOUT augmentation
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),  # Auto-converts [0-255] to [0-1]
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print("[02-FEATURE-EXTRACTION] Transformations configured:")
print("  Train transform:")
print("    1. Resize(224, 224)")
print("    2. RandomHorizontalFlip(p=0.5)")
print("    3. RandomRotation(±15°)")
print("    4. ColorJitter(brightness/contrast/saturation/hue)")
print("    5. ToTensor() → [0-255] to [0-1]")
print("    6. Normalize(ImageNet mean/std)")
print("\n  Val/Test transform:")
print("    1. Resize(224, 224)")
print("    2. ToTensor() → [0-255] to [0-1]")
print("    3. Normalize(ImageNet mean/std)")
print("    (NO augmentation!)")

## Custom Dataset Class

In [ ]:
class WasteDataset(Dataset):
    """
    Custom dataset untuk load images dari CSV paths.
    Images di-load on-the-fly, TIDAK di-cache!
    """
    def __init__(self, df, dataset_base_path, transform=None):
        self.df = df.reset_index(drop=True)
        self.dataset_base_path = dataset_base_path
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image from path
        img_path = row['image_path']
        
        # Handle path (Kaggle vs local)
        if not os.path.exists(img_path):
            # Try relative path from dataset_base_path
            class_name = row['class_name']
            filename = os.path.basename(img_path)
            img_path = os.path.join(self.dataset_base_path, class_name, filename)
        
        # Load and convert to RGB
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        label = row['class_idx']
        
        return image, label, img_path

print("[02-FEATURE-EXTRACTION] WasteDataset class defined")
print("  Features:")
print("    - Load images on-the-fly (tidak di-cache)")
print("    - Augmentasi realtime saat __getitem__()")
print("    - Support Kaggle + local paths")

## MobileNetV3 Feature Extractor

In [ ]:
class MobileNetV3FeatureExtractor(nn.Module):
    """
    MobileNetV3-Large feature extractor.
    Extract 960-dim features BEFORE classifier layer.
    """
    def __init__(self):
        super().__init__()
        
        # Load pre-trained MobileNetV3-Large
        model = models.mobilenet_v3_large(pretrained=True)
        
        # Extract feature layers (sebelum classifier)
        self.features = model.features
        self.avgpool = model.avgpool
        self.feature_dim = 960  # MobileNetV3-Large output
        
        # Freeze all parameters (no training)
        for param in self.parameters():
            param.requires_grad = False
            
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = x.flatten(1)
        return x

# Initialize extractor
print("[02-FEATURE-EXTRACTION] Loading MobileNetV3-Large...")
feature_extractor = MobileNetV3FeatureExtractor().to(device)
feature_extractor.eval()

print(f"[02-FEATURE-EXTRACTION] MobileNetV3 loaded:")
print(f"  Feature dimension: {feature_extractor.feature_dim}")
print(f"  Pretrained: ImageNet")
print(f"  Mode: eval() - no training")

## Feature Extraction Function

In [ ]:
def extract_features(model, dataloader, split_name):
    """
    Extract CNN features dari DataLoader.
    Augmentasi terjadi di dalam loop ini (realtime)!
    """
    model.eval()
    features = []
    labels = []
    paths = []
    
    with torch.no_grad():
        for images, label_batch, path_batch in tqdm(dataloader, desc=f'Extracting {split_name}'):
            images = images.to(device)
            
            # Extract features (images sudah di-augment di DataLoader!)
            feat = model(images)
            
            features.append(feat.cpu().numpy())
            labels.extend(label_batch.numpy())
            paths.extend(path_batch)
    
    features = np.vstack(features)
    labels = np.array(labels)
    
    print(f"\n[02-FEATURE-EXTRACTION] {split_name} extraction complete:")
    print(f"  Shape: {features.shape}")
    print(f"  Mean: {features.mean():.4f}, Std: {features.std():.4f}")
    print(f"  Min: {features.min():.4f}, Max: {features.max():.4f}")
    
    return features, labels, paths

print("[02-FEATURE-EXTRACTION] Extraction function ready")

## Extract Features for All Splits

In [ ]:
splits = {'train': train_df, 'val': val_df, 'test': test_df}
results = {}

print("[02-FEATURE-EXTRACTION] Starting feature extraction...")
print("="*60)

for split_name, df in splits.items():
    print(f"\nProcessing {split_name} split ({len(df):,} samples)...")
    
    if split_name == 'train':
        current_transform = train_transform
        print(f"  Transform: WITH augmentation (RandomFlip, Rotation, ColorJitter)")
    else:
        current_transform = val_test_transform
        print(f"  Transform: WITHOUT augmentation (Resize + Normalize only)")
    
    # Create dataset with appropriate transform
    dataset = WasteDataset(df, DATASET_BASE_PATH, transform=current_transform)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)
    
    # Extract features (augmentation happens in DataLoader loop!)
    features, labels, paths = extract_features(feature_extractor, dataloader, split_name)
    
    results[split_name] = {
        'features': features,
        'labels': labels,
        'paths': paths
    }

print("\n" + "="*60)
print("[02-FEATURE-EXTRACTION] All extractions completed!")

## Feature Normalization (StandardScaler)

In [ ]:
print("[02-FEATURE-EXTRACTION] Normalizing features with StandardScaler...")

# Initialize scaler
scaler = StandardScaler()

# Fit on train features ONLY
train_features_normalized = scaler.fit_transform(results['train']['features'])

# Transform val/test (NO fit!)
val_features_normalized = scaler.transform(results['val']['features'])
test_features_normalized = scaler.transform(results['test']['features'])

print(f"\n[02-FEATURE-EXTRACTION] Normalization complete:")
print(f"  Train features normalized:")
print(f"    Shape: {train_features_normalized.shape}")
print(f"    Mean: {train_features_normalized.mean():.6f} (should be ~0)")
print(f"    Std:  {train_features_normalized.std():.6f} (should be ~1)")

print(f"\n  Val features normalized:")
print(f"    Shape: {val_features_normalized.shape}")
print(f"    Mean: {val_features_normalized.mean():.6f}")
print(f"    Std:  {val_features_normalized.std():.6f}")

print(f"\n  Test features normalized:")
print(f"    Shape: {test_features_normalized.shape}")
print(f"    Mean: {test_features_normalized.mean():.6f}")
print(f"    Std:  {test_features_normalized.std():.6f}")

# Update results
results['train']['features_normalized'] = train_features_normalized
results['val']['features_normalized'] = val_features_normalized
results['test']['features_normalized'] = test_features_normalized

## Save Normalized Features + Scaler

In [ ]:
print(f"[02-FEATURE-EXTRACTION] Saving to: {OUTPUT_DIR}")

# Save normalized features
np.save(f'{OUTPUT_DIR}/train_mobilenet_features.npy', train_features_normalized)
np.save(f'{OUTPUT_DIR}/val_mobilenet_features.npy', val_features_normalized)
np.save(f'{OUTPUT_DIR}/test_mobilenet_features.npy', test_features_normalized)

print(f"[02-FEATURE-EXTRACTION] Features saved:")
print(f"  ✓ train_mobilenet_features.npy ({train_features_normalized.shape})")
print(f"  ✓ val_mobilenet_features.npy ({val_features_normalized.shape})")
print(f"  ✓ test_mobilenet_features.npy ({test_features_normalized.shape})")

# Save labels
np.save(f'{OUTPUT_DIR}/train_labels.npy', results['train']['labels'])
np.save(f'{OUTPUT_DIR}/val_labels.npy', results['val']['labels'])
np.save(f'{OUTPUT_DIR}/test_labels.npy', results['test']['labels'])

print(f"  ✓ train/val/test_labels.npy")

# Save paths
pd.DataFrame({'path': results['train']['paths']}).to_csv(f'{OUTPUT_DIR}/train_paths.csv', index=False)
pd.DataFrame({'path': results['val']['paths']}).to_csv(f'{OUTPUT_DIR}/val_paths.csv', index=False)
pd.DataFrame({'path': results['test']['paths']}).to_csv(f'{OUTPUT_DIR}/test_paths.csv', index=False)

print(f"  ✓ train/val/test_paths.csv")

# Save scaler (CRITICAL for inference!)
joblib.dump(scaler, f'{OUTPUT_DIR}/mobilenet_scaler.pkl')
print(f"  ✓ mobilenet_scaler.pkl (REQUIRED for production!)")

# Save metadata
metadata = {
    'model': 'MobileNetV3-Large',
    'feature_dim': 960,
    'train_size': len(train_features_normalized),
    'val_size': len(val_features_normalized),
    'test_size': len(test_features_normalized),
    'augmentation': {
        'train': 'RandomHorizontalFlip, RandomRotation, ColorJitter',
        'val_test': 'None'
    },
    'normalization': {
        'image': 'ImageNet (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])',
        'features': 'StandardScaler (Z-score)'
    }
}

with open(f'{OUTPUT_DIR}/feature_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"  ✓ feature_metadata.json")

# List all saved files
print(f"\n[02-FEATURE-EXTRACTION] All files saved:")
for file in sorted(os.listdir(OUTPUT_DIR)):
    file_path = os.path.join(OUTPUT_DIR, file)
    if os.path.isfile(file_path):
        file_size = os.path.getsize(file_path) / (1024 * 1024)
        print(f"  {file} ({file_size:.2f} MB)")

## Feature Extraction Summary

In [ ]:
summary = f"""
========================================
02. FEATURE EXTRACTION SUMMARY - JakOlah
========================================

CNN Architecture:
  Model: MobileNetV3-Large (pretrained on ImageNet)
  Feature dimension: 960
  Mode: Inference only (no training)

Data Augmentation (PyTorch Best Practice):
  Train Set: WITH augmentation (realtime in DataLoader)
    - RandomHorizontalFlip(p=0.5)
    - RandomRotation(±15°)
    - ColorJitter(brightness/contrast/saturation/hue)
  
  Val/Test Set: WITHOUT augmentation
    - Resize + Normalize only

Preprocessing Pipeline:
  1. Resize → 224x224
  2. Augmentation (train only, NOT saved to disk!)
  3. ToTensor → [0-255] to [0-1] (automatic)
  4. Normalize → ImageNet mean/std
  5. CNN Extraction → 960-dim features
  6. StandardScaler → Z-score normalization

Dataset Statistics:
  Train: {len(train_features_normalized):,} samples
  Val:   {len(val_features_normalized):,} samples
  Test:  {len(test_features_normalized):,} samples

Feature Statistics (After StandardScaler):
  Train mean: {train_features_normalized.mean():.6f} (target: 0.0)
  Train std:  {train_features_normalized.std():.6f} (target: 1.0)

Output Files:
  ✓ train_mobilenet_features.npy (normalized)
  ✓ val_mobilenet_features.npy (normalized)
  ✓ test_mobilenet_features.npy (normalized)
  ✓ train/val/test_labels.npy
  ✓ train/val/test_paths.csv
  ✓ mobilenet_scaler.pkl (CRITICAL for inference!)
  ✓ feature_metadata.json

CRITICAL NOTES:
  1. Augmented images are NOT saved (realtime processing)
  2. Features are ALREADY NORMALIZED (don't normalize again in 03!)
  3. mobilenet_scaler.pkl is REQUIRED for production deployment
  4. Val/Test features use scaler.transform() (not fit_transform)

Production Inference Pipeline:
  Image → Resize → Normalize(ImageNet) → MobileNetV3 → scaler.transform() → SVM

========================================
NEXT STEP: Run 03-SVM-Training-Step.ipynb
========================================
"""

print(summary)

with open(f'{OUTPUT_DIR}/feature_extraction_summary.md', 'w', encoding='utf-8') as f:
    f.write(summary)

print(f"[02-FEATURE-EXTRACTION] Summary saved to {OUTPUT_DIR}/feature_extraction_summary.md")
print(f"[02-FEATURE-EXTRACTION] ✅ COMPLETED")

## Download Output

In [ ]:
import shutil

# Create zip file of all outputs
zip_filename = '02-feature-extraction-output'
shutil.make_archive(zip_filename, 'zip', OUTPUT_DIR)

print(f"[02-FEATURE-EXTRACTION] Output zipped to: {zip_filename}.zip")
print(f"  File size: {os.path.getsize(f'{zip_filename}.zip') / (1024*1024):.2f} MB")
print(f"\n💾 Download {zip_filename}.zip dari Kaggle output panel")